# Batch correction with CellANOVA

CellANOVA (Zhang et al., *Nat Biotechnol* 2024) is a **variance-decomposition** approach: given a control pool of cells expected to be biologically homogeneous across batches, it estimates the per-batch nuisance variance and removes it from the rest of the data. Especially strong when you can designate a control compartment that shouldn't change across conditions.

This is one of the **omicverse batch-correction zoo** tutorials. For an overview of all backends, the recommendation tree, and the unified `_BATCH_OBSM` schema, see [batch/index](../index.md). For the side-by-side comparison of all backends on the NeurIPS 2021 multi-batch benchmark, see [t_single_batch](../t_single_batch.ipynb).

Optional dependency: `pip install cellanova` (also bundled in `omicverse[integration]`).

**Setup step**: CellANOVA requires a control pool. Set `adata.uns['control_dict']` to a dict mapping pool name → list of batch ids before the call.


## Load a multi-batch dataset

We use the same toy multi-batch AnnData as the other zoo tutorials — three NeurIPS 2021 batches concatenated. Replace with your own dataset by changing `adata` and `batch_key` below.

In [ ]:
import omicverse as ov
import scanpy as sc
import numpy as np

# Replace these URLs with your own dataset; the three are
# the same multi-batch dataset used by t_single_batch.
adata1 = ov.datasets.get_adata(
    'https://figshare.com/ndownloader/files/41932005',
    filename='neurips2021_s1d3.h5ad',
)
adata2 = ov.datasets.get_adata(
    'https://figshare.com/ndownloader/files/41932008',
    filename='neurips2021_s2d1.h5ad',
)
adata3 = ov.datasets.get_adata(
    'https://figshare.com/ndownloader/files/41932011',
    filename='neurips2021_s3d7.h5ad',
)
adata = sc.concat([adata1, adata2, adata3], merge='same')
adata.obs['batch'] = adata.obs['batch'].astype('category')
adata

## Preprocess + PCA (shared across all backends)

Every backend in the zoo starts from the same QC'd, log-normalised AnnData with `scaled|original|X_pca` in obsm. Read [t_single_batch](../t_single_batch.ipynb) for the full discussion of these steps.

In [ ]:
adata = ov.pp.qc(adata,
                 tresh={'mito_perc': 0.2, 'nUMIs': 500,
                        'detected_genes': 250})
ov.utils.store_layers(adata, layers='counts')
adata = ov.pp.preprocess(adata, mode='shiftlog|pearson',
                         n_HVGs=2000, batch_key=None)
adata.raw = adata
adata = adata[:, adata.var.highly_variable_features]
ov.pp.scale(adata)
ov.pp.pca(adata, layer='scaled', n_pcs=50)

## Run `ov.single.batch_correction(methods='CellANOVA')`

The wrapper routes method-specific kwargs to the right destination — for scvi-tools backends this includes splitting between `__init__` (architecture) and `.train()` (optimisation). See the **Key parameters** section below.

In [ ]:
ov.single.batch_correction(
    adata,
    batch_key='batch',
    methods='CellANOVA',
    # CellANOVA needs adata.uns['control_dict'] populated
    # before the call — see the t_single_batch overview for the
    # control-pool construction recipe.
    integrate_key='batch',
)

## Visualise the corrected embedding

Every backend writes its corrected representation to a stable obsm key — for **CellANOVA** it is `adata.obsm['X_cellanova']`. We project it via `ov.utils.mde` for a lightweight UMAP-style display.

In [ ]:
adata.obsm['X_mde_cellanova'] = ov.utils.mde(adata.obsm['X_cellanova'])
ov.pl.embedding(
    adata,
    basis='X_mde_cellanova',
    color=['batch'],
    frameon='small',
    title='CellANOVA — coloured by batch',
)

## Key parameters

- `integrate_key` — usually the same as `batch_key`.
- CellANOVA writes a denoised expression matrix into `adata.layers['denoised']` alongside the embedding.


## Related tutorials

- `combat` — when no control pool is available.
- `harmony` — when you only need embedding-level correction.

For a side-by-side comparison of every backend on the same benchmark + scib-metrics scoring, see [../t_single_batch](../t_single_batch.ipynb).